In [3]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

# Set up paths relative to the repo root
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
# Load the CSV (change filename if yours is different)
crosswalk_raw = pd.read_csv(DATA_RAW / 'nist_800_53_rev5_hipaa_crosswalk.csv')
crosswalk_raw.head()

,Focal Document\nElement,\nFocal Document Element Description,Security Control\nBaseline,Reference Document\nElement,Reference Document\nElement Description,Fulfilled By (Y/N),Group Identifier\n(optional),Comments\n(optional),Strength of\nRelationship\n(optional)
0,AC-01,"a. Develop, document, and disseminate to [Assi...",Low,164.308(a)(3),Workforce Security: Implement policies and pro...,NaN,NaN,NaN,NaN
1,AC-01,"a. Develop, document, and disseminate to [Assi...",Low,164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,NaN,NaN,NaN,NaN
2,AC-01,"a. Develop, document, and disseminate to [Assi...",Low,164.308(a)(4),Information Access Management: Implement polic...,NaN,NaN,NaN,NaN
3,AC-01,"a. Develop, document, and disseminate to [Assi...",Low,164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,NaN,NaN,NaN,NaN
4,AC-01,"a. Develop, document, and disseminate to [Assi...",Low,164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,NaN,NaN,NaN,NaN


In [6]:
# Step 1: Clean the column names by removing newlines and extra spaces
crosswalk_raw.columns = (
    crosswalk_raw.columns
    .str.replace('\n', ' ', regex=False)   # replace newline with space
    .str.replace(r'\s+', ' ', regex=True)   # collapse multiple spaces
    .str.strip()                            # remove leading/trailing spaces
)

# Check the cleaned column names
print("Cleaned column names:", crosswalk_raw.columns.tolist())

# Step 2: Map to our desired names
column_map = {
    'Focal Document Element': 'nist_control_id',
    'Focal Document Element Description': 'nist_text',
    'Reference Document Element': 'hipaa_citation',
    'Reference Document Element Description': 'hipaa_text',
    'Fulfilled By (Y/N)': 'fulfilled_by',
    'Strength of Relationship (optional)': 'strength_of_relationship'
}
crosswalk = crosswalk_raw.rename(columns=column_map)

# Step 3: Keep only the needed columns
crosswalk = crosswalk[['nist_control_id', 'nist_text', 'hipaa_citation', 'hipaa_text', 'fulfilled_by', 'strength_of_relationship']]

crosswalk.head()

Cleaned column names: ['Focal Document Element', 'Focal Document Element Description', 'Security Control Baseline', 'Reference Document Element', 'Reference Document Element Description', 'Fulfilled By (Y/N)', 'Group Identifier (optional)', 'Comments (optional)', 'Strength of Relationship (optional)']


,nist_control_id,nist_text,hipaa_citation,hipaa_text,fulfilled_by,strength_of_relationship
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,NaN,NaN
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,NaN,NaN
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,NaN,NaN
3,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,NaN,NaN
4,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,NaN,NaN


In [7]:
# Drop rows where either control or citation is missing
crosswalk.dropna(subset=['nist_control_id', 'hipaa_citation'], inplace=True)

# Clean whitespace
crosswalk['nist_control_id'] = crosswalk['nist_control_id'].str.strip()
crosswalk['hipaa_citation'] = crosswalk['hipaa_citation'].str.strip()

# Remove duplicate pairs (keep the first mapping type)
crosswalk = crosswalk.drop_duplicates(subset=['nist_control_id', 'hipaa_citation'])
print(f"Clean crosswalk pairs: {len(crosswalk)}")
crosswalk.head()

Clean crosswalk pairs: 40


,nist_control_id,nist_text,hipaa_citation,hipaa_text,fulfilled_by,strength_of_relationship
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,NaN,NaN
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,NaN,NaN
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,NaN,NaN
3,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,NaN,NaN
4,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,NaN,NaN


In [8]:
# Drop rows where text is missing or HIPAA citation is NaN
crosswalk.dropna(subset=['nist_text', 'hipaa_text', 'hipaa_citation'], inplace=True)

# Clean whitespace in text columns
for col in ['nist_text', 'hipaa_text', 'hipaa_citation']:
    crosswalk[col] = crosswalk[col].astype(str).str.strip()

# Remove duplicate control-citation pairs (keep first)
crosswalk = crosswalk.drop_duplicates(subset=['nist_control_id', 'hipaa_citation'])

# Create binary label: Y = 1, else 0
def get_binary_label(val):
    if isinstance(val, str) and val.strip().upper() == 'Y':
        return 1
    return 0

crosswalk['label'] = crosswalk['fulfilled_by'].apply(get_binary_label)

print(f"Cleaned dataset: {len(crosswalk)} rows")
print("Label distribution:")
print(crosswalk['label'].value_counts())
crosswalk.head()

Cleaned dataset: 40 rows
Label distribution:
label
0    40
Name: count, dtype: int64


,nist_control_id,nist_text,hipaa_citation,hipaa_text,fulfilled_by,strength_of_relationship,label
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,NaN,NaN,0
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,NaN,NaN,0
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,NaN,NaN,0
3,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,NaN,NaN,0
4,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,NaN,NaN,0


In [12]:
labeled = crosswalk.copy()
labeled.to_csv(DATA_PROCESSED / 'labeled_pairs.csv', index=False)
print("Saved labeled_pairs.csv")

Saved labeled_pairs.csv


In [14]:
#Run this in powershell.
#git add .
#git commit -m "Remove unnecessary NIST/HIPAA download cells; data prep complete with crosswalk"
#git push origin main

SyntaxError: invalid syntax (3990918869.py, line 1)

In [11]:
controls = []
for group in nist_catalog['catalog']['groups']:
    for control in group.get('controls', []):
        cid = control['id']
        title = control.get('title', '')
        # The full text is in 'parts' – we'll combine the statement parts
        text_parts = []
        for part in control.get('parts', []):
            if part['name'] == 'statement':
                # Extract prose from the statement part
                for subpart in part.get('parts', []):
                    prose = subpart.get('prose', '') if 'prose' in subpart else ''
                    # Some subparts have 'parts' with more prose
                    if not prose and 'parts' in subpart:
                        prose = ' '.join(p.get('prose','') for p in subpart['parts'] if 'prose' in p)
                    text_parts.append(prose)
        full_text = ' '.join(text_parts).strip()
        controls.append({'control_id': cid, 'text': full_text})

nist_df = pd.DataFrame(controls)
print(f"Extracted {len(nist_df)} NIST controls")
nist_df.head()

NameError: name 'nist_catalog' is not defined

In [ ]:
# eCFR API endpoint for Title 45, Part 164, Subpart C (Security Standards)
# We'll request the whole Part 164 and later filter
ecfr_url = "https://www.ecfr.gov/api/versioner/v1/full/2025-05-02/title-45.xml?part=164"  
# Note: The date might need to be recent; the API date should be a valid snapshot.
# You can also use the "latest" date by calling the versioner endpoint first.
# For simplicity, we'll try a known date. If it fails, we'll use a fallback.

resp = requests.get(ecfr_url)
if resp.status_code != 200:
    # Fallback to a known good date
    ecfr_url = "https://www.ecfr.gov/api/versioner/v1/full/2024-01-01/title-45.xml?part=164"
    resp = requests.get(ecfr_url)

with open(DATA_RAW / 'title45_part164.xml', 'w', encoding='utf-8') as f:
    f.write(resp.text)

print("Downloaded HIPAA Security Rule XML.")